# Building Data Generation

This notebook shows how to generate building data files for different buildings, locations, weather years, internal gain profiles, and time windows.

Each saved file contains simulated building sensor/state data, controller output, heat-pump/comfort metrics, and the disturbances used for the simulation. Disturbances are generated internally but are not saved as standalone files.

## Setup

Weather loading is handled by `src.disturbances.load_weather`.

- If a matching local DWD TRY file exists in `data/weather` and `year=2015` is requested, it is used locally.
- Otherwise, cached PVGIS JSON files are used when available.
- If no cache exists, PVGIS is queried online and the response is cached in `data/weather`.

For the bundled Freiburg coordinates, `year=2015` uses the local TRY2015 file, while other years fall back to PVGIS for year-specific weather.

I do not see closest-weather-station fallback logic in the current `disturbances.py`; the current fallback is PVGIS. Random city locations are available through `disturbances.get_random_location`, exposed here as `location="random_DE"`.

For uncached locations or years, this notebook needs network access.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in notebooks/, so the repo root is one level up.
I4B_ROOT = Path.cwd().resolve()
if I4B_ROOT.name == "notebooks":
    I4B_ROOT = I4B_ROOT.parent

if str(I4B_ROOT) not in sys.path:
    sys.path.insert(0, str(I4B_ROOT))

import data.buildings as building_catalog
from src import data_generation as dg

OUTPUT_DIR = I4B_ROOT / "data" / "generated" / "building_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {I4B_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## Available Inputs

In [ ]:
available_buildings = list(building_catalog.__all__)
available_profiles = sorted((I4B_ROOT / "data" / "profiles" / "InternalGains").glob("*.csv"))

print(f"{len(available_buildings)} buildings")
print(available_buildings)
print("\nInternal gain profiles")
for profile in available_profiles:
    print("-", profile.name)

Define locations, weather years, and the export time window. A location can be the building's default location, an explicit override, or a random city sampled by `disturbances.get_random_location` through names such as `random_DE`.

The time window is interpreted as `[START_DATE, END_DATE)`: start is included, end is excluded. This makes one-week or one-month ranges easy to express, for example `2015-01-01` to `2015-02-01`.

In [ ]:
LOCATIONS = {
    **dg.DEFAULT_LOCATIONS,
    # Add project-specific locations here.
    "custom_example": {
        "lat": 52.52,
        "long": 13.405,
        "altitude": 34,
        "timezone": "Europe/Berlin",
    },
}

YEARS = [2015]
TIMESTEP_SECONDS = 900
START_DATE = "2015-01-01"
END_DATE = "2015-01-08"

# Use one of LOCATIONS keys, "building_default", or strings like "random_DE".
LOCATION = "freiburg"

LOCATIONS

## Module Helpers

The reusable implementation lives in `src/data_generation.py`. The notebook calls `dg.generate_building_data_file(...)`, which creates one CSV and one metadata JSON file for a single scenario.

The CSV includes:

- simulated building states such as `T_room`, `T_wall`, `T_hp_ret`, depending on the selected model
- controller output `T_hp_sup`
- disturbances `T_amb`, `Qdot_gains`, `Qdot_sol`, `Qdot_int`
- comfort and heat-pump metrics such as `E_el`, `P_el`, `COP`, `Qdot_th`, `dev_neg_max`
- scenario columns such as building, location, year, profile, method, and controller

In [ ]:
help(dg.generate_building_data_file)

## Generate One Building Data File

In [ ]:
csv_path, metadata_path, dataset, metadata, results = dg.generate_building_data_file(
    building_name="sfh_1984_1994_1_enev",
    location=LOCATION,
    year=2015,
    profile_name="ResidentialDetached.csv",
    method="4R3C",
    timestep_seconds=900,
    start_date=START_DATE,
    end_date=END_DATE,
    hp_model_name="Heatpump_AW",
    ctrl_method="heatcurve",
    initial_temperature=20.0,
    night_setback=True,
    output_dir=OUTPUT_DIR,
    locations=LOCATIONS,
    repo_filepath=I4B_ROOT,
)

print(csv_path)
print(metadata_path)
display(dataset.head())
display(dataset.describe())

## Generate A Scenario Matrix

Adjust these lists to create a dataset matrix. Every scenario saves one building-data CSV and one metadata JSON. If a weather/location/year combination is not cached, the first run may download PVGIS data.

For random locations, set `SCENARIO_LOCATIONS = ["random_DE"]`. The existing random-location function samples a city from Overpass; weather then follows the current `load_weather` behavior.

In [ ]:
SCENARIO_BUILDINGS = [
    "sfh_1979_1983_0_soc",
    "sfh_1984_1994_1_enev",
    "sfh_2002_2009_2_kfw",
]
SCENARIO_LOCATIONS = [LOCATION]
SCENARIO_PROFILES = ["ResidentialDetached.csv", "Office.csv"]
SCENARIO_YEARS = [2015]

saved = []
for building_name in SCENARIO_BUILDINGS:
    for location in SCENARIO_LOCATIONS:
        for profile_name in SCENARIO_PROFILES:
            for year in SCENARIO_YEARS:
                csv_path, metadata_path, dataset, metadata, _ = dg.generate_building_data_file(
                    building_name=building_name,
                    location=location,
                    year=year,
                    profile_name=profile_name,
                    method="4R3C",
                    timestep_seconds=3600,
                    start_date=START_DATE,
                    end_date=END_DATE,
                    hp_model_name="Heatpump_AW",
                    ctrl_method="heatcurve",
                    initial_temperature=20.0,
                    night_setback=True,
                    output_dir=OUTPUT_DIR,
                    locations=LOCATIONS,
                    repo_filepath=I4B_ROOT,
                )
                saved.append(metadata | {"csv_path": str(csv_path), "metadata_path": str(metadata_path)})

summary = pd.DataFrame(saved)
summary

## Inspect Sensor And Disturbance Columns

The saved file includes building states and the disturbances used by the simulator. The exact state columns depend on the selected RC model.

In [ ]:
disturbance_columns = ["T_amb", "Qdot_gains", "Qdot_sol", "Qdot_int"]
sensor_columns = [
    col for col in dataset.columns
    if col.startswith("T_") and col not in disturbance_columns + ["T_room_set_lower"]
]
energy_columns = ["P_el", "E_el", "COP", "Qdot_th"]
comfort_columns = ["dev_neg_sum", "dev_neg_max", "dev_pos_sum", "dev_pos_max"]

display(dataset[sensor_columns + disturbance_columns + energy_columns + comfort_columns].head(24))

## Load A Saved Building Data File

In [ ]:
loaded = pd.read_csv(csv_path, index_col="datetime", parse_dates=True)
print(loaded.shape)
display(loaded.head())